### install the neccessary packages 


*   transformers - To use huggingface transformers "t5_small" model
*   evaluate - A library for easily evaluating the model
*   datasets - A library for easily accessing and sharing datasets
*   rouge-score - A set of metrics and a software package used for evaluating automatic summarization




In [1]:
!pip install transformers
!pip install evaluate
!pip install datasets
!pip install rouge_score

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 45.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 52.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.8/199.8 KB 10.5 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 KB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.7/468.7 KB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 KB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.2/212.2 KB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 KB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 KB 11.7 MB/s eta 0:00:00



*   installing torch




In [ ]:
!pip install torch

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


### Login to huggingface hub
 with a token in order to be able to push the model to the hub after training.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Token is valid.
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful


### import classes:

*   AutoTokenizer - a generic tokenizer class
*   DataCollatorForSeq2Seq - Data collator that will dynamically pad the inputs received, as well as the labels.
*   AutoModelForSeq2SeqLM - a generic model class that will be instantiated as one of the model classes of the library (with a sequence-to-sequence language modeling head)
*   pipeline - used for inference
*   files and io - used for uploading the dataset




In [2]:
from transformers import AutoTokenizer
from transformers import DataCollatorForSeq2Seq
import evaluate
import numpy as np
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import pipeline
import pandas as pd
from datasets import Dataset
from google.colab import files
import io

### upload the dataset
MeQSum is a dataset of 1000 rows containing medical questions and their summarized version.




In [3]:
# upload the dataset 
uploaded = files.upload()
# read the MeQSum.csv file
df = pd.read_csv(io.BytesIO(uploaded["MeQSum.csv"]), encoding="ISO-8859-1")
df = pd.DataFrame(df)
# print length of dataframe
print(len(df))


Saving MeQSum.csv to MeQSum.csv
1000


### split the dataset
split the dataset into train and test sets. the first 900 rows are used for training and the other 100 for testing the model.

In [4]:
# use first 900 rows for training 
train_df = df.iloc[:900, :]
# use the rest (100 rows) for testing
test_df = df.iloc[900:, :]
# split the data into train and test
train_ds = Dataset.from_pandas(train_df, split="train")
test_ds = Dataset.from_pandas(test_df, split="test")
# print length of train and test datasets
print(len(train_ds))
print(len(test_ds))

900
100


### Preproccess the data
add "summarize" to each question as a prefix(to let t5 know the task is summarization) and tokenize each question and it's summarization using AutoTokenizer class. truncate text to be no more than the max_length

In [5]:
# tokenize the text
checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# specify a prefix to let the model (t5_small) know the task is summarization
prefix = "summarize: "


def preprocess_function(examples):
    # add the prefix to questions
    inputs = [prefix + doc for doc in examples["CHQ"]]
    # tokenize questions while truncating them to be no more than 128 in length
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    # tokenize summaries and truncate them to be no more than 32 in length
    labels = tokenizer(text_target=examples["Summary"], max_length=32, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


/usr/local/lib/python3.9/dist-packages/transformers/models/t5/tokenization_t5_fast.py:155: FutureWarning: This tokenizer was incorrectly instantiated with a model max length of 512 which will be corrected in Transformers v5.
For now, this behavior is kept to avoid breaking backwards compatibility when padding/encoding with `truncation is True`.
- Be aware that you SHOULD NOT rely on t5-small automatically truncating your input to 512 when padding/encoding.
- If you want to encode/pad to sequences longer than 512 you can either instantiate this tokenizer with `model_max_length` or pass `max_length` when encoding/padding.
- To avoid this warning, please instantiate this tokenizer with `model_max_length` set to your preferred value.
  warnings.warn(


In [6]:
# tokenize whole train and test dataset by calling preprocess_function for each of them
tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_test = test_ds.map(preprocess_function, batched=True)
# print length of tokenized training set
print(len(tokenized_train))


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

900


### Padding
use DataCollatorForSeq2Seq to add paddings to the text to the longest length.

In [ ]:
# padding the dataset
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)


### Evaluate
use rouge score to evaluate the model while training based on predictions and summaries provided.

In [ ]:
#evaluate using rouge metric
rouge = evaluate.load("rouge")

# compute the rouge metric based on predictions and summaries provided in dataset
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # decode labels
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # compute rouge based on decoded labels and predictions
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}


### Train the model 
train the model (50 epochs) with specified arguments, tokenizer, evaluzation metric,.. using AutoModelForSeq2SeqLM class.


as we see the following is the result:

Training Loss=1.375200,	Validation Loss=1.186928,	Rouge1=0.494200,	Rouge2=0.346000,	Rougel=0.474300,	Rougelsum=0.474000,	Gen Len=13.610000



							

In [ ]:

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
# specify the arguments to train the model with
training_args = Seq2SeqTrainingArguments(
    output_dir="meQ_model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=50,
    predict_with_generate=True,
    logging_strategy="epoch",
    # fp16=True,
    push_to_hub=True,
)
# specify the model, arguments, train and test data, tokenizer, data_collator(used for padding) and evaluation metrics for training  
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,

)
# train the model
trainer.train()


/content/meQ_model is already a clone of https://huggingface.co/TaniyaHaghighi/meQ_model. Make sure you pull the latest changes with `repo.git_pull()`.
/usr/local/lib/python3.9/dist-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,2.835000,2.047524,0.234500,0.089500,0.207400,0.206700,16.820000
2,2.380600,1.724254,0.285400,0.126000,0.264900,0.266700,15.830000
3,2.170100,1.565917,0.363300,0.211000,0.344500,0.344400,14.720000
4,2.019400,1.468990,0.430100,0.275200,0.411500,0.412700,13.950000
5,1.921800,1.415901,0.468000,0.312500,0.449900,0.449300,13.260000
6,1.864500,1.381104,0.487000,0.346200,0.472000,0.471500,13.220000
7,1.825400,1.349399,0.483600,0.342400,0.466800,0.466000,13.170000
8,1.784300,1.327724,0.481800,0.344400,0.467600,0.467200,13.040000
9,1.754300,1.308375,0.477500,0.341200,0.463000,0.462700,13.150000
10,1.720600,1.296065,0.476000,0.339300,0.460200,0.461700,13.130000


TrainOutput(global_step=2850, training_loss=1.5768531130071273, metrics={'train_runtime': 24689.8703, 'train_samples_per_second': 1.823, 'train_steps_per_second': 0.115, 'total_flos': 1519797497757696.0, 'train_loss': 1.5768531130071273, 'epoch': 50.0})

### push to hub
the next step is to push the model to huggingface hub.

In [ ]:
trainer.push_to_hub()

Upload file pytorch_model.bin:   0%|          | 1.00/231M [00:00<?, ?B/s]

Upload file runs/Apr01_13-36-40_ef107563e904/events.out.tfevents.1680356207.ef107563e904.208.10:   0%|        …

To https://huggingface.co/TaniyaHaghighi/meQ_model
   2c79432..3b1bed9  main -> main

   2c79432..3b1bed9  main -> main

To https://huggingface.co/TaniyaHaghighi/meQ_model
   3b1bed9..8e66b42  main -> main

   3b1bed9..8e66b42  main -> main



'https://huggingface.co/TaniyaHaghighi/meQ_model/commit/3b1bed935a682f2395946b2bbc46fc25e07a69ff'

### Inference
use medical questions and the model will predict the summarization for the question.

In [ ]:
# a medical question to be summarized
text = "SUBJECT: Neuropathy Pain in Legs and Feet:I have neuropathy in both feet and legs. They feel heavy and at night they sometimes burn. My feet feel like I have a second skin, which makes it difficult to wiggle my toes and arch my feet."
# using pipeline to get the summarized question.
summarizer = pipeline("summarization", model="TaniyaHaghighi/meQ_model", max_length=100)
# print the summarization
print(summarizer(text))

Your max_length is set to 100, but you input_length is only 66. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=33)


[{'summary_text': 'What are the treatments for leg pain? . . . . . . . . . . .'}]


some examples:

SUBJECT: Neuropathy Pain in Legs and Feet:I have neuropathy in both feet and legs. They feel heavy and at night they sometimes burn. My feet feel like I have a second skin, which makes it difficult to wiggle my toes and arch my feet.

summary:
What are the treatments for leg pain? . . . . 

---------------------------

SUBJECT: Lactose Intolerant Child?: My child was diagnosed with lactose intolerance. Dairy makes her sick but she can eat lactose-free dairy products without issue. Is there something else that should be explored, or is my child a rare one year old with true lactose intolerance?
summary:
What are the treatments for lactose intolerance? . Are there other treatments for it? . What are the treatments for it?

---------------------------

SUBJECT: Numbness and Tingling in the Hands:I wake up with my right hand numb. My thumb and middle finger seem the worst. It takes awhile for the numbness to wear off. I work in a grocery store, so I'm always on my feet and on the register. What should I do?
summary:
What are the treatments for numbness? . (i) What are the treatments for numbness? ? (i) What are the treatments for numbness?